In [ ]:
from google.colab import files
import pandas as pd
import io

# Upload the red wine CSV
uploaded = files.upload()

# Automatically get whatever filename was uploaded
filename = list(uploaded.keys())[0]

print("Uploaded filename:", filename)

# Read directly from the uploaded file
df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=";")

# Check dataset
print("\nDataset shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())

display(df.head())

Saving winequality-red.csv to winequality-red.csv
Uploaded filename: winequality-red.csv

Dataset shape: (1359, 12)
Duplicates: 0
Missing values: 0


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5


In [ ]:
X = df.drop("quality", axis=1)
y = df["quality"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nQuality distribution:")
print(y.value_counts().sort_index())

X shape: (1359, 11)
y shape: (1359,)

Features:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']

Quality distribution:
quality
3     10
4     53
5    577
6    535
7    167
8     17
Name: count, dtype: int64


Evaluation framework

In [ ]:
from sklearn.model_selection import RepeatedKFold

# 5-fold cross-validation repeated 3 times
cv = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

print("Total evaluation folds:", cv.get_n_splits())

Total evaluation folds: 15


mean-predictor baseline

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from scipy.stats import spearmanr
import numpy as np

# Spearman correlation scorer
def spearman_score(y_true, y_pred):
    return spearmanr(y_true, y_pred).statistic

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "Spearman": make_scorer(spearman_score)
}

# Mean-predictor baseline
baseline = DummyRegressor(strategy="mean")

baseline_results = cross_validate(
    baseline,
    X,
    y,
    cv=cv,
    scoring=scoring
)

# Convert negative sklearn error scores to positive errors
mae = -baseline_results["test_MAE"]
rmse = -baseline_results["test_RMSE"]
r2 = baseline_results["test_R2"]
spearman = baseline_results["test_Spearman"]

print("RED WINE — MEAN PREDICTOR BASELINE")
print("----------------------------------")
print(f"MAE:      {mae.mean():.4f} ± {mae.std():.4f}")
print(f"RMSE:     {rmse.mean():.4f} ± {rmse.std():.4f}")
print(f"R²:       {r2.mean():.4f} ± {r2.std():.4f}")
print(f"Spearman: {np.nanmean(spearman):.4f} ± {np.nanstd(spearman):.4f}")

/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defi

RED WINE — MEAN PREDICTOR BASELINE
----------------------------------
MAE:      0.6949 ± 0.0188
RMSE:     0.8234 ± 0.0230
R²:       -0.0034 ± 0.0037
Spearman: nan ± nan


/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(y_true, y_pred).statistic
/tmp/ipykernel_1559/290002323.py:40: RuntimeWarning: Mean of empty slice
  print(f"Spearman: {np.nanmean(spearman):.4f} ± {np.nanstd(spearman):.4f}")
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


StandardScaler → Ridge Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_validate, KFold
from sklearn.metrics import make_scorer
from scipy.stats import spearmanr
import numpy as np

# Pipeline: scaling happens inside CV
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# Hyperparameters to test
ridge_params = {
    "ridge__alpha": [0.01, 0.1, 1, 10, 100]
}

# Inner CV: used ONLY for hyperparameter selection
inner_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid search
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_params,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    n_jobs=-1
)

# Evaluation metrics
def spearman_score(y_true, y_pred):
    return spearmanr(y_true, y_pred).statistic

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "Spearman": make_scorer(spearman_score)
}

# Outer CV = the 5-fold × 3-repeat CV we already created
ridge_results = cross_validate(
    ridge_search,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

# Convert error scores to positive values
ridge_mae = -ridge_results["test_MAE"]
ridge_rmse = -ridge_results["test_RMSE"]
ridge_r2 = ridge_results["test_R2"]
ridge_spearman = ridge_results["test_Spearman"]

print("RED WINE — RIDGE REGRESSION")
print("--------------------------------")
print(f"MAE:      {ridge_mae.mean():.4f} ± {ridge_mae.std():.4f}")
print(f"RMSE:     {ridge_rmse.mean():.4f} ± {ridge_rmse.std():.4f}")
print(f"R²:       {ridge_r2.mean():.4f} ± {ridge_r2.std():.4f}")
print(f"Spearman: {ridge_spearman.mean():.4f} ± {ridge_spearman.std():.4f}")

print("\nBest alpha selected in each outer fold:")
best_alphas = [
    est.best_params_["ridge__alpha"]
    for est in ridge_results["estimator"]
]
print(best_alphas)

RED WINE — RIDGE REGRESSION
--------------------------------
MAE:      0.5135 ± 0.0170
RMSE:     0.6644 ± 0.0183
R²:       0.3453 ± 0.0437
Spearman: 0.5924 ± 0.0326

Best alpha selected in each outer fold:
[10, 100, 10, 10, 100, 100, 10, 10, 100, 10, 100, 10, 10, 100, 10]


Next: Model 2 — KNN Regressor

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_validate

# KNN pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

# Hyperparameters
knn_params = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"]
}

# Inner tuning
knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_params,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    n_jobs=-1
)

# Outer evaluation
knn_results = cross_validate(
    knn_search,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

# Extract metrics
knn_mae = -knn_results["test_MAE"]
knn_rmse = -knn_results["test_RMSE"]
knn_r2 = knn_results["test_R2"]
knn_spearman = knn_results["test_Spearman"]

print("RED WINE — KNN REGRESSOR")
print("--------------------------------")
print(f"MAE:      {knn_mae.mean():.4f} ± {knn_mae.std():.4f}")
print(f"RMSE:     {knn_rmse.mean():.4f} ± {knn_rmse.std():.4f}")
print(f"R²:       {knn_r2.mean():.4f} ± {knn_r2.std():.4f}")
print(f"Spearman: {knn_spearman.mean():.4f} ± {knn_spearman.std():.4f}")

print("\nBest KNN parameters selected in each outer fold:")

best_knn_params = [
    est.best_params_
    for est in knn_results["estimator"]
]

for i, params in enumerate(best_knn_params, start=1):
    print(f"Fold {i}: {params}")

RED WINE — KNN REGRESSOR
--------------------------------
MAE:      0.5180 ± 0.0163
RMSE:     0.6690 ± 0.0157
R²:       0.3363 ± 0.0386
Spearman: 0.5906 ± 0.0284

Best KNN parameters selected in each outer fold:
Fold 1: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 2: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 3: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 4: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 5: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 6: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 7: {'knn__n_neighbors': 11, 'knn__weights': 'distance'}
Fold 8: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 9: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 10: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 11: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 12: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Fold 13: {'knn__n_neighbors': 15, 'knn__weights': 'di

Model 3 — Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_validate

# Random Forest
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

# Hyperparameter grid
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2]
}

# Inner CV hyperparameter tuning
rf_search = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    n_jobs=-1
)

# Outer repeated cross-validation
rf_results = cross_validate(
    rf_search,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

# Extract metrics
rf_mae = -rf_results["test_MAE"]
rf_rmse = -rf_results["test_RMSE"]
rf_r2 = rf_results["test_R2"]
rf_spearman = rf_results["test_Spearman"]

print("RED WINE — RANDOM FOREST REGRESSOR")
print("------------------------------------")
print(f"MAE:      {rf_mae.mean():.4f} ± {rf_mae.std():.4f}")
print(f"RMSE:     {rf_rmse.mean():.4f} ± {rf_rmse.std():.4f}")
print(f"R²:       {rf_r2.mean():.4f} ± {rf_r2.std():.4f}")
print(f"Spearman: {rf_spearman.mean():.4f} ± {rf_spearman.std():.4f}")

print("\nBest Random Forest parameters selected in each outer fold:")

best_rf_params = [
    est.best_params_
    for est in rf_results["estimator"]
]

for i, params in enumerate(best_rf_params, start=1):
    print(f"Fold {i}: {params}")

KeyboardInterrupt: 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_validate

# Random Forest
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Smaller tuning grid
rf_params = {
    "n_estimators": [100],
    "max_depth": [None, 10],
    "min_samples_leaf": [1, 2]
}

# Inner hyperparameter tuning
rf_search = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    scoring="neg_root_mean_squared_error",
    cv=3,                 # 3-fold INNER CV
    n_jobs=1
)

# Outer evaluation:
# keep our required 5-fold × 3 repeats
rf_results = cross_validate(
    rf_search,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

# Results
rf_mae = -rf_results["test_MAE"]
rf_rmse = -rf_results["test_RMSE"]
rf_r2 = rf_results["test_R2"]
rf_spearman = rf_results["test_Spearman"]

print("RED WINE — RANDOM FOREST REGRESSOR")
print("------------------------------------")
print(f"MAE:      {rf_mae.mean():.4f} ± {rf_mae.std():.4f}")
print(f"RMSE:     {rf_rmse.mean():.4f} ± {rf_rmse.std():.4f}")
print(f"R²:       {rf_r2.mean():.4f} ± {rf_r2.std():.4f}")
print(f"Spearman: {rf_spearman.mean():.4f} ± {rf_spearman.std():.4f}")

print("\nBest parameters selected in each outer fold:")

for i, est in enumerate(rf_results["estimator"], 1):
    print(f"Fold {i}: {est.best_params_}")

RED WINE — RANDOM FOREST REGRESSOR
------------------------------------
MAE:      0.4954 ± 0.0171
RMSE:     0.6443 ± 0.0174
R²:       0.3841 ± 0.0437
Spearman: 0.6237 ± 0.0303

Best parameters selected in each outer fold:
Fold 1: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 2: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 3: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 4: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 5: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 6: {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 7: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 100}
Fold 8: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 9: {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 10: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fold 11: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 100}
Fo

Model 4 — SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_validate

# Scaling is important for SVR
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf"))
])

# Small tuning grid
svr_params = {
    "svr__C": [1, 10],
    "svr__epsilon": [0.1, 0.2]
}

# Inner tuning
svr_search = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=svr_params,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

# Outer evaluation: same 5-fold × 3 repeats
svr_results = cross_validate(
    svr_search,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

# Extract metrics
svr_mae = -svr_results["test_MAE"]
svr_rmse = -svr_results["test_RMSE"]
svr_r2 = svr_results["test_R2"]
svr_spearman = svr_results["test_Spearman"]

print("RED WINE — SVR")
print("--------------------------------")
print(f"MAE:      {svr_mae.mean():.4f} ± {svr_mae.std():.4f}")
print(f"RMSE:     {svr_rmse.mean():.4f} ± {svr_rmse.std():.4f}")
print(f"R²:       {svr_r2.mean():.4f} ± {svr_r2.std():.4f}")
print(f"Spearman: {svr_spearman.mean():.4f} ± {svr_spearman.std():.4f}")

print("\nBest SVR parameters selected in each outer fold:")

for i, est in enumerate(svr_results["estimator"], 1):
    print(f"Fold {i}: {est.best_params_}")

RED WINE — SVR
--------------------------------
MAE:      0.4963 ± 0.0176
RMSE:     0.6539 ± 0.0186
R²:       0.3663 ± 0.0346
Spearman: 0.6114 ± 0.0232

Best SVR parameters selected in each outer fold:
Fold 1: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 2: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 3: {'svr__C': 1, 'svr__epsilon': 0.1}
Fold 4: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 5: {'svr__C': 1, 'svr__epsilon': 0.1}
Fold 6: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 7: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 8: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 9: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 10: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 11: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 12: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 13: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 14: {'svr__C': 1, 'svr__epsilon': 0.2}
Fold 15: {'svr__C': 1, 'svr__epsilon': 0.2}


In [ ]:
import pandas as pd
import numpy as np

n_folds = len(ridge_mae)

red_fold_results = pd.DataFrame({
    "Fold": np.arange(1, n_folds + 1),

    "Ridge_MAE": ridge_mae,
    "Ridge_RMSE": ridge_rmse,
    "Ridge_R2": ridge_r2,
    "Ridge_Spearman": ridge_spearman,

    "KNN_MAE": knn_mae,
    "KNN_RMSE": knn_rmse,
    "KNN_R2": knn_r2,
    "KNN_Spearman": knn_spearman,

    "RF_MAE": rf_mae,
    "RF_RMSE": rf_rmse,
    "RF_R2": rf_r2,
    "RF_Spearman": rf_spearman,

    "SVR_MAE": svr_mae,
    "SVR_RMSE": svr_rmse,
    "SVR_R2": svr_r2,
    "SVR_Spearman": svr_spearman
})

red_fold_results.to_csv(
    "red_wine_fold_results.csv",
    index=False
)

red_fold_results

,Fold,Ridge_MAE,Ridge_RMSE,Ridge_R2,Ridge_Spearman,KNN_MAE,KNN_RMSE,KNN_R2,KNN_Spearman,RF_MAE,RF_RMSE,RF_R2,RF_Spearman,SVR_MAE,SVR_RMSE,SVR_R2,SVR_Spearman
0,1,0.503789,0.655396,0.393603,0.643814,0.508038,0.656365,0.391810,0.643011,0.469354,0.620410,0.456618,0.679150,0.476252,0.629607,0.440387,0.659499
1,2,0.493860,0.650782,0.342383,0.603086,0.489142,0.641573,0.360862,0.611096,0.491854,0.641557,0.360894,0.626182,0.477416,0.638896,0.366185,0.630056
2,3,0.521461,0.668712,0.353832,0.567973,0.527930,0.676449,0.338792,0.570000,0.496429,0.642143,0.404158,0.614227,0.492175,0.664268,0.362391,0.596003
3,4,0.534239,0.679102,0.315411,0.568600,0.537227,0.693585,0.285899,0.555150,0.505297,0.649311,0.374158,0.612222,0.512228,0.671272,0.331107,0.594336
4,5,0.507498,0.664711,0.335112,0.583909,0.519898,0.666688,0.331151,0.589732,0.513872,0.663227,0.338077,0.588114,0.494303,0.667667,0.329184,0.582139
5,6,0.534849,0.676111,0.393123,0.616194,0.528694,0.661736,0.418656,0.613150,0.513173,0.658239,0.424784,0.635936,0.525466,0.671523,0.401331,0.609594
6,7,0.517802,0.680672,0.227141,0.537666,0.489340,0.665238,0.261793,0.554653,0.484263,0.645734,0.304444,0.595503,0.476582,0.640924,0.314769,0.594444
7,8,0.513006,0.680647,0.348824,0.606226,0.515254,0.675946,0.357788,0.625303,0.493096,0.651980,0.402519,0.625028,0.515199,0.670999,0.367154,0.619547
8,9,0.536118,0.678625,0.311274,0.550317,0.536573,0.672023,0.324609,0.564406,0.511528,0.648579,0.370910,0.620295,0.499569,0.644434,0.378925,0.624490
9,10,0.470695,0.610891,0.423271,0.653238,0.499341,0.642074,0.362891,0.631077,0.457467,0.604049,0.436118,0.671277,0.466219,0.619827,0.406276,0.647297


In [ ]:
from scipy.stats import friedmanchisquare

statistic, p_value = friedmanchisquare(
    ridge_rmse,
    knn_rmse,
    rf_rmse,
    svr_rmse
)

print("FRIEDMAN TEST — RED WINE RMSE")
print("--------------------------------")
print(f"Statistic: {statistic:.4f}")
print(f"p-value:   {p_value:.6f}")

if p_value < 0.05:
    print("Result: Significant differences exist among the algorithms.")
else:
    print("Result: No statistically significant difference was detected among the algorithms.")

FRIEDMAN TEST — RED WINE RMSE
--------------------------------
Statistic: 25.3200
p-value:   0.000013
Result: Significant differences exist among the algorithms.


In [ ]:
from scipy.stats import wilcoxon
from itertools import combinations
import pandas as pd

models = {
    "Ridge": ridge_rmse,
    "KNN": knn_rmse,
    "Random Forest": rf_rmse,
    "SVR": svr_rmse
}

pairwise_results = []

# Perform pairwise Wilcoxon tests
for model1, model2 in combinations(models.keys(), 2):

    stat, p = wilcoxon(
        models[model1],
        models[model2],
        alternative="two-sided"
    )

    pairwise_results.append({
        "Model 1": model1,
        "Model 2": model2,
        "Wilcoxon Statistic": stat,
        "Raw p-value": p
    })

pairwise_df = pd.DataFrame(pairwise_results)

# Holm correction
m = len(pairwise_df)

sorted_indices = pairwise_df["Raw p-value"].argsort()
sorted_p = pairwise_df.loc[
    sorted_indices, "Raw p-value"
].values

holm_adjusted = []

for i, p in enumerate(sorted_p):
    adjusted_p = min((m - i) * p, 1.0)
    holm_adjusted.append(adjusted_p)

# Enforce monotonicity
for i in range(1, len(holm_adjusted)):
    holm_adjusted[i] = max(
        holm_adjusted[i],
        holm_adjusted[i-1]
    )

pairwise_df.loc[
    sorted_indices, "Holm Adjusted p-value"
] = holm_adjusted

pairwise_df["Significant (α=0.05)"] = (
    pairwise_df["Holm Adjusted p-value"] < 0.05
)

pairwise_df

,Model 1,Model 2,Wilcoxon Statistic,Raw p-value,Holm Adjusted p-value,Significant (α=0.05)
0,Ridge,KNN,37.0,0.207764,0.207764,False
1,Ridge,Random Forest,2.0,0.000183,0.000916,True
2,Ridge,SVR,19.0,0.018066,0.036133,True
3,KNN,Random Forest,0.0,0.000061,0.000366,True
4,KNN,SVR,5.0,0.000610,0.002441,True
5,Random Forest,SVR,12.0,0.004272,0.012817,True


save

In [ ]:
pairwise_df.to_csv(
    "red_wine_wilcoxon_posthoc.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


COST

In [ ]:
import pandas as pd

cost_results = pd.DataFrame({
    "Model": ["Ridge", "KNN", "Random Forest", "SVR"],

    "Mean Fit Time (s)": [
        ridge_results["fit_time"].mean(),
        knn_results["fit_time"].mean(),
        rf_results["fit_time"].mean(),
        svr_results["fit_time"].mean()
    ],

    "SD Fit Time (s)": [
        ridge_results["fit_time"].std(),
        knn_results["fit_time"].std(),
        rf_results["fit_time"].std(),
        svr_results["fit_time"].std()
    ],

    "Mean Score Time (s)": [
        ridge_results["score_time"].mean(),
        knn_results["score_time"].mean(),
        rf_results["score_time"].mean(),
        svr_results["score_time"].mean()
    ]
})

print("RED WINE — COMPUTATIONAL COST")
display(cost_results)

cost_results.to_csv(
    "red_wine_computational_cost.csv",
    index=False
)

RED WINE — COMPUTATIONAL COST


,Model,Mean Fit Time (s),SD Fit Time (s),Mean Score Time (s)
0,Ridge,0.541889,0.140875,0.017143
1,KNN,2.018340,0.781529,0.050167
2,Random Forest,13.900893,1.807671,0.066324
3,SVR,2.819091,0.944467,0.059466
